# r2x spike S3: totals step
Throwaway. Total amount per region from the `sales` input, written to the `totals` output. Values come from the helper: passed in by the framework, or from the sandbox section of `process.yml` when run by hand.

In [0]:
%pip install -q /Volumes/sconset-db-01/r2x_spike/libraries/r2x_helper-0.1.0-py3-none-any.whl pyyaml==6.0.3

In [0]:
import r2x_helper
v = r2x_helper.get_values()
print(v)

In [0]:
from calc import totals_by_region  # plain .py file next to this notebook

if v.settings["fail_step"] == "totals":
    raise RuntimeError("totals: failing on purpose (spike setting fail_step)")

sales = spark.sql(f"SELECT * FROM {v.input_ref('sales')}")
totals_by_region(sales, v.settings["min_amount"]).createOrReplaceTempView("totals")
spark.sql("INSERT INTO IDENTIFIER(:out) SELECT :rid, region, total, current_timestamp() FROM totals",
          args={"out": v.outputs["totals"], "rid": v.request_id})
rows = spark.table("totals").count()

In [0]:
import json, importlib.metadata as md
out = {"step": "totals", "source": v.source, "request_id": v.request_id, "rows_written": rows,
       "wrote_to": v.outputs["totals"], "read": v.input_ref("sales"),
       "versions": {"r2x_helper": r2x_helper.__version__, "pyyaml": md.version("pyyaml")}}
print(json.dumps(out, indent=2))
dbutils.notebook.exit(json.dumps(out))